# Clase 3 · 03 · Deduplicación con estado y watermark
Un archivo puede ser nuevo y contener un evento ya recibido. Deduplicamos por `event_id`,
con estado limitado por event time. La garantía cubre el horizonte configurado, no todo el historial indefinidamente.

In [ ]:
dbutils.widgets.text("student_id", "alumno01", "01 - Identificador")
dbutils.widgets.dropdown("scale", "small", ["test", "small", "demo"], "02 - Escala de clases 1 y 2")
dbutils.widgets.dropdown("expected_batch_id", "stream_001",
                         ["stream_001", "stream_002", "stream_003", "stream_004", "stream_005"], "03 - Etapa a validar")
dbutils.widgets.text("job_run_id", "interactive", "04 - ID de ejecución")

In [ ]:
%run ../common/config

In [ ]:
%run ../common/generate_stream_data

In [ ]:
%run ../common/streaming_support

In [ ]:
from pyspark.sql import functions as F
config = CourseConfig(spark.sql("SELECT current_catalog()").first()[0],
                      dbutils.widgets.get("student_id"), dbutils.widgets.get("scale"))
repetitions = STREAM_REPETITIONS[config.scale]
expected_batch_id = normalize_stream_batch(dbutils.widgets.get("expected_batch_id"))
job_run_id = dbutils.widgets.get("job_run_id").strip()[:100] or "interactive"
spark.conf.set("spark.sql.session.timeZone", "UTC")
require_stream_lab(spark, config)

## 1. Preparar el flujo válido
El watermark es de 10 minutos sobre `event_ts`. No es una espera de reloj de pared.
El experimento controla el orden de llegadas y valida cada etapa antes de publicar la siguiente.
Cada operator tiene su propio estado: esta consulta y Gold usan checkpoints separados.

In [ ]:
valid = (spark.readStream.table(qualified_table(config, "silver_stream_classified"))
         .where(F.col("quality_reason").isNull())
         .select("event_id", "event_ts", "customer_id", "product_id", "amount", "payment_channel",
                 "is_fraud", "event_type", "source_batch_id", "country", "category"))
unique_events = (valid.withWatermark("event_ts", "10 minutes")
                 .dropDuplicatesWithinWatermark(["event_id"]))
assert unique_events.isStreaming

## 2. Persistir eventos únicos
`e002` aparece en 001 y 002: se emite una vez. `late_ok` llega con event time 12:03 y se acepta.
Tras procesar 003, el máximo event time es 12:26; un evento a 12:02 de 004 queda fuera del horizonte.
Conservamos esa entrada en Bronze y Silver valid, aunque no pase al flujo deduplicado.

In [ ]:
query = (unique_events.writeStream.format("delta").outputMode("append")
         .option("checkpointLocation", stream_checkpoint(config, "dedup"))
         .queryName(f"{config.schema}_dedup")
         .trigger(availableNow=True)
         .toTable(qualified_table(config, "silver_stream_events")))
wait_and_record_stream(spark, config, query, "dedup", job_run_id)

In [ ]:
%sql
SELECT event_type, COUNT(*) AS rows, COUNT(DISTINCT event_id) AS distinct_ids
FROM silver_stream_events GROUP BY event_type ORDER BY event_type

In [ ]:
%sql
SELECT event_id, event_ts, source_batch_id FROM silver_stream_valid
WHERE event_id IN ('e002_000', 'late_ok_000', 'late_bad_000') ORDER BY event_id, source_batch_id

### Para observar
`numRowsDroppedByWatermark` en el progreso de dedup indica descartes por tiempo.
La cuarentena conserva problemas de contrato; no recibe automáticamente eventos descartados por el operador de estado.